In [ ]:
# Rail Fare Structural Reform Analysis

This notebook models fare increases with a structural reform that distributes the difference between baseline and reform proportionally to households based on their `rail_subsidy_spending`.

## Fare Increase Schedule:
- **Baseline**: 2026 (+5.8%), 2027 (+4.2%), 2028 (+3.9%), 2029 (+3.9%)
- **Reform**: 2026 (+5.8% × 1.02), 2027 (+4.2%), 2028 (+3.9%), 2029 (+3.9%)

In [1]:
from policyengine_uk import Microsimulation, Scenario
from policyengine_uk.data import UKSingleYearDataset
import pandas as pd
import numpy as np

# Load local dataset
dataset = UKSingleYearDataset(
    file_path="/Users/janansadeqian/policyengine-uk-data/policyengine_uk_data/storage/enhanced_frs_2023_24.h5"
)

/Users/janansadeqian/anaconda3/envs/python313/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Fare increase rates
FARE_INCREASES = {
    2026: 0.058,  # 5.8%
    2027: 0.042,  # 4.2%
    2028: 0.039,  # 3.9%
    2029: 0.039,  # 3.9%
}

# Reform multiplier for 2026 (1.058 × 1.02)
REFORM_2026_MULTIPLIER = 1.02

**Data Sources:**
- [GOV.UK Rail Fares Freeze Impact](https://www.gov.uk/government/publications/rail-fares-freeze-passenger-savings-estimate) - 2026 counterfactual (5.8%)
- [OBR March 2025 Economic Outlook](https://obr.uk/efo/economic-and-fiscal-outlook-march-2025/) - RPI forecasts (3.2%, 2.9%)
- [GOV.UK Railways Bill Factsheet](https://www.gov.uk/government/publications/railways-bill/railways-bill-fares) - fare formula (July RPI + 1%)

## Baseline Scenario

Standard fare increases compounding year over year.

In [3]:
def apply_baseline_fare_increases(sim):
    """
    Baseline: Apply standard fare increases each year.
    Starting from 2025 base, compound the increases.
    """
    # Get 2025 baseline values
    base_2025 = sim.calculate("rail_subsidy_spending", 2025, map_to="household")
    
    # Calculate cumulative multipliers
    cumulative = 1.0
    
    for year in [2026, 2027, 2028, 2029]:
        cumulative *= (1 + FARE_INCREASES[year])
        
        # Apply to each household proportionally
        new_values = base_2025 * cumulative
        sim.set_input("rail_subsidy_spending", year, new_values)
    
    return sim

baseline_scenario = Scenario(simulation_modifier=apply_baseline_fare_increases)

## Reform Scenario

2026 gets an additional multiplier (1.02), then standard increases continue.

In [4]:
def apply_reform_fare_increases(sim):
    """
    Reform: 2026 gets multiplied by (1.058 × 1.02), then continue normally.
    Distribute excess proportionally based on household share of baseline subsidy.
    """
    # Get 2025 baseline values
    base_2025 = sim.calculate("rail_subsidy_spending", 2025, map_to="household")
    
    # Track cumulative multipliers
    baseline_cumulative = 1.0
    reform_cumulative = 1.0
    
    for year in [2026, 2027, 2028, 2029]:
        # Calculate baseline multiplier (standard fare increases)
        baseline_cumulative *= (1 + FARE_INCREASES[year])
        
        # Calculate reform multiplier (2026 gets extra 1.02)
        if year == 2026:
            reform_cumulative = (1 + FARE_INCREASES[year]) * REFORM_2026_MULTIPLIER
        else:
            reform_cumulative *= (1 + FARE_INCREASES[year])
        
        # Calculate what baseline would be (per household)
        baseline_subsidy_hh = base_2025 * baseline_cumulative
        
        # Calculate aggregate totals
        baseline_total = baseline_subsidy_hh.sum()
        reform_total_target = base_2025.sum() * reform_cumulative
        total_excess = reform_total_target - baseline_total
        
        # Distribute excess proportionally
        household_share = baseline_subsidy_hh / baseline_total
        household_gain = total_excess * household_share
        
        # Set reformed subsidy
        reformed_subsidy = baseline_subsidy_hh + household_gain
        sim.set_input("rail_subsidy_spending", year, reformed_subsidy)
    
    return sim

reform_scenario = Scenario(simulation_modifier=apply_reform_fare_increases)

In [5]:
# Create microsimulations
baseline = Microsimulation(dataset=dataset, scenario=baseline_scenario)
reformed = Microsimulation(dataset=dataset, scenario=reform_scenario)

## Revenue Comparison Table

In [6]:
# Compare baseline vs reformed rail subsidy spending
years = [2025, 2026, 2027, 2028, 2029]

print("=== Rail Fare Revenue Comparison ===")
print()
print(f"{'Year':<8} {'Baseline':>12} {'Reform':>12} {'Difference':>12} {'% Change':>10}")
print("=" * 60)

for year in years:
    baseline_spending = baseline.calculate("rail_subsidy_spending", period=year).sum() / 1e9
    reform_spending = reformed.calculate("rail_subsidy_spending", period=year).sum() / 1e9
    difference = reform_spending - baseline_spending
    pct_change = (difference / baseline_spending) * 100 if baseline_spending > 0 else 0
    
    print(f"{year:<8} £{baseline_spending:>10.2f}bn £{reform_spending:>10.2f}bn £{difference:>10.2f}bn {pct_change:>9.2f}%")

=== Rail Fare Revenue Comparison ===

Year         Baseline       Reform   Difference   % Change
2025     £      7.67bn £      7.67bn £      0.00bn      0.00%
2026     £      8.14bn £      8.31bn £      0.16bn      2.00%
2027     £      8.52bn £      8.69bn £      0.17bn      2.00%
2028     £      8.89bn £      9.06bn £      0.18bn      2.00%
2029     £      9.27bn £      9.46bn £      0.19bn      2.00%


## Verification: Sum of Household Distribution = Aggregate Difference

This checks that the sum of all amounts distributed to households equals the exact aggregate difference between reform and baseline.

In [7]:
# Verification: Check that household-level distribution sums to aggregate difference
print("=== Verification: Distribution Check ===")
print()
print(f"{'Year':<8} {'Aggregate Diff':>15} {'Sum of HH Dist':>15} {'Match':>10}")
print("=" * 50)

for year in years:
    # Get household-level arrays
    baseline_array = baseline.calculate("rail_subsidy_spending", period=year)
    reform_array = reformed.calculate("rail_subsidy_spending", period=year)
    
    # Calculate difference at household level
    household_differences = reform_array - baseline_array
    
    # Sum the household differences (with weights)
    sum_of_distributed = household_differences.sum()
    
    # Calculate aggregate difference
    aggregate_difference = reform_array.sum() - baseline_array.sum()
    
    # Check if they match
    match = "✓ Yes" if abs(sum_of_distributed - aggregate_difference) < 0.01 else "✗ No"
    
    print(f"{year:<8} £{aggregate_difference / 1e9:>13.2f}bn £{sum_of_distributed / 1e9:>13.2f}bn {match:>10}")

=== Verification: Distribution Check ===

Year      Aggregate Diff  Sum of HH Dist      Match
2025     £         0.00bn £         0.00bn      ✓ Yes
2026     £         0.16bn £         0.16bn      ✓ Yes
2027     £         0.17bn £         0.17bn      ✓ Yes
2028     £         0.18bn £         0.18bn      ✓ Yes
2029     £         0.19bn £         0.19bn      ✓ Yes


## Key Insights

- **Baseline**: Fares compound at standard rates starting from 2025
- **Reform**: 2026 includes an additional 2% multiplier, creating a permanent level shift
- **Distribution**: The difference is distributed proportionally to households based on their baseline `rail_subsidy_spending`

## How Distribution Works (with Household Weights)

1. **Per-Household Arrays**: `sim.calculate("rail_subsidy_spending", year)` returns an array with one value per household
2. **Proportional Scaling**: Each household's value is multiplied by the same factor, preserving their relative share
3. **Automatic Weighting**: When `.sum()` is called, PolicyEngine automatically applies household weights to calculate population totals
4. **Result**: Households with higher baseline `rail_subsidy_spending` receive proportionally more of the additional subsidy

## Impact by Income Decile

This section analyzes how the rail fare reform affects households across the income distribution:
- Average change in household net income by decile
- Relative change as percentage of income
- Total impact on each decile

Note: Decile -1 (households with negative net income) is excluded from this analysis.

In [8]:
# Load household-level data for 2026
year = 2026

vars_to_load = [
    'household_id',
    'household_net_income',
    'household_weight',
    'rail_subsidy_spending',
]

# Load from baseline
household_dict_baseline = {}
for var in vars_to_load:
    household_dict_baseline[var] = baseline.calculate(var, year, map_to="household").values

# Load from reformed
household_dict_reformed = {}
for var in vars_to_load:
    household_dict_reformed[var] = reformed.calculate(var, year, map_to="household").values

# Create dataframes
baseline_hh_df = pd.DataFrame(household_dict_baseline)
reformed_hh_df = pd.DataFrame(household_dict_reformed)

# Calculate rail subsidy gain
baseline_hh_df['reformed_rail_subsidy'] = reformed_hh_df['rail_subsidy_spending']
baseline_hh_df['rail_subsidy_gain'] = baseline_hh_df['reformed_rail_subsidy'] - baseline_hh_df['rail_subsidy_spending']
baseline_hh_df['income_change'] = baseline_hh_df['rail_subsidy_gain']
baseline_hh_df['relative_change'] = 100 * baseline_hh_df['income_change'] / baseline_hh_df['household_net_income']

# Filter out households with negative net income
baseline_hh_df = baseline_hh_df[baseline_hh_df['household_net_income'] > 0].copy()

# Create proper income deciles based on household net income (weighted)
baseline_hh_df = baseline_hh_df.sort_values('household_net_income').reset_index(drop=True)
baseline_hh_df['cumulative_weight'] = baseline_hh_df['household_weight'].cumsum()
total_weight = baseline_hh_df['household_weight'].sum()

baseline_hh_df['income_decile'] = pd.cut(
    baseline_hh_df['cumulative_weight'],
    bins=[0] + [total_weight * i / 10 for i in range(1, 11)],
    labels=range(1, 11),
    include_lowest=True
).astype(int)

baseline_decile_df = baseline_hh_df.copy()

print(f"Loaded {len(baseline_decile_df):,} households with positive income")
print(f"Assigned to deciles 1-10 based on weighted household net income")

Loaded 50,876 households with positive income
Assigned to deciles 1-10 based on weighted household net income


In [9]:
# Calculate aggregate statistics
total_households = (baseline_decile_df['household_weight']).sum()
households_with_gain = baseline_decile_df[baseline_decile_df['income_change'] > 0.01]
households_with_gain_count = (households_with_gain['household_weight']).sum()

# Calculate aggregates using household weights
total_rail_subsidy_gain = (baseline_decile_df['rail_subsidy_gain'] * baseline_decile_df['household_weight']).sum()
total_income_change = (baseline_decile_df['income_change'] * baseline_decile_df['household_weight']).sum()
avg_income_change = total_income_change / total_households

print("=" * 70)
print("AGGREGATE STATISTICS (2026, All Households)")
print("=" * 70)
print(f"Total households:       {total_households:,.0f}")
print(f"Households with gain:   {households_with_gain_count:,.0f} ({100 * households_with_gain_count / total_households:.2f}%)")
print(f"\nRail subsidy gain:      £{total_rail_subsidy_gain / 1e9:.2f} bn")
print(f"Income change:          £{total_income_change / 1e9:.2f} bn")
print(f"Average per household:  £{avg_income_change:,.2f}")
print(f"\nVerification: {'✓ Match' if abs(total_income_change - total_rail_subsidy_gain) < 1 else '✗ No match'}") 

# Aggregate by decile
decile_summary = baseline_decile_df.groupby('income_decile').apply(
    lambda x: pd.Series({
        'total_households': (x['household_weight']).sum(),
        'avg_baseline_income': (x['household_net_income'] * x['household_weight']).sum() / (x['household_weight']).sum(),
        'avg_income_change': (x['income_change'] * x['household_weight']).sum() / (x['household_weight']).sum(),
        'relative_change_pct': (x['income_change'] * x['household_weight']).sum() / (x['household_net_income'] * x['household_weight']).sum() * 100,
        'total_change': (x['income_change'] * x['household_weight']).sum(),
    }), include_groups=False
).reset_index()

# Print decile impact table
print("\n=== Rail Fare Reform Impact by Income Decile (2026) ===")
print()
print(f"{'Decile':<8} {'Avg Income':>12} {'Avg Change':>12} {'% Change':>10} {'Total Impact':>14}")
print("=" * 60)

for _, row in decile_summary.iterrows():
    decile = int(row['income_decile'])
    avg_income = row['avg_baseline_income']
    avg_change = row['avg_income_change']
    pct_change = row['relative_change_pct']
    total_change = row['total_change']
    
    print(f"{decile:<8} £{avg_income:>10,.0f} £{avg_change:>10,.2f} {pct_change:>9.3f}% £{total_change/1e9:>12.2f}bn")

# Total
total_change_all = (baseline_decile_df['income_change'] * baseline_decile_df['household_weight']).sum()
avg_change_all = total_change_all / (baseline_decile_df['household_weight']).sum()
print("=" * 60)
print(f"{'TOTAL':<8} {'':<12} £{avg_change_all:>10,.2f} {'':<10} £{total_change_all/1e9:>12.2f}bn")

AGGREGATE STATISTICS (2026, All Households)
Total households:       31,480,496
Households with gain:   853,165 (2.71%)

Rail subsidy gain:      £0.15 bn
Income change:          £0.15 bn
Average per household:  £4.87

Verification: ✓ Match

=== Rail Fare Reform Impact by Income Decile (2026) ===

Decile     Avg Income   Avg Change   % Change   Total Impact
1        £    11,689 £      2.58     0.022% £        0.01bn
2        £    21,510 £      1.28     0.006% £        0.00bn
3        £    27,520 £      1.00     0.004% £        0.00bn
4        £    33,196 £      1.97     0.006% £        0.01bn
5        £    40,257 £      3.07     0.008% £        0.01bn
6        £    47,923 £      2.76     0.006% £        0.01bn
7        £    56,104 £      5.58     0.010% £        0.02bn
8        £    66,262 £      6.54     0.010% £        0.02bn
9        £    81,336 £      5.02     0.006% £        0.02bn
10       £   172,287 £     18.89     0.011% £        0.06bn
TOTAL                 £      4.87         

In [10]:
# Verification: Check that total income change matches expected rail subsidy difference
print("\n=== Verification: Income Change vs Expected Subsidy Difference ===")

# Calculate expected difference based on 2025 baseline
rail_subsidy_2025 = baseline.calculate("rail_subsidy_spending", 2025, map_to="household").sum()

# Expected 2026 baseline: 2025 * 1.058
expected_baseline_2026 = rail_subsidy_2025 * (1 + FARE_INCREASES[2026])

# Expected 2026 reform: 2025 * 1.058 * 1.02
expected_reform_2026 = rail_subsidy_2025 * (1 + FARE_INCREASES[2026]) * REFORM_2026_MULTIPLIER

# Expected difference
expected_difference_2026 = expected_reform_2026 - expected_baseline_2026

# Total income change from decile table
total_change_from_deciles = (baseline_decile_df['income_change'] * baseline_decile_df['household_weight']).sum()

print(f"2025 baseline rail subsidy:       £{rail_subsidy_2025 / 1e9:.2f} bn")
print(f"Expected 2026 baseline:           £{expected_baseline_2026 / 1e9:.2f} bn")
print(f"Expected 2026 reform:             £{expected_reform_2026 / 1e9:.2f} bn")
print(f"Expected difference:              £{expected_difference_2026 / 1e9:.2f} bn")
print(f"\nTotal income change (deciles):    £{total_change_from_deciles / 1e9:.2f} bn")
print(f"\nMatch: {'✓ Yes' if abs(expected_difference_2026 - total_change_from_deciles) / 1e9 < 0.01 else '✗ No'}")
print(f"Difference: £{abs(expected_difference_2026 - total_change_from_deciles) / 1e6:.2f} million")


=== Verification: Income Change vs Expected Subsidy Difference ===
2025 baseline rail subsidy:       £7.67 bn
Expected 2026 baseline:           £8.11 bn
Expected 2026 reform:             £8.28 bn
Expected difference:              £0.16 bn

Total income change (deciles):    £0.15 bn

Match: ✓ Yes
Difference: £9.06 million


## Summary

This notebook models a rail fare reform where:

1. **Baseline**: Rail subsidy spending increases by standard fare increase rates (5.8%, 4.2%, 3.9%, 3.9%)
2. **Reform**: 2026 gets an additional 2% multiplier (1.058 × 1.02), then continues with standard rates
3. **Distribution**: The additional subsidy (reform - baseline) is distributed to households proportionally based on their baseline `rail_subsidy_spending`, with household weights applied
4. **Household Benefit**: The additional subsidy increases household net income directly
5. **Decile Impact**: The analysis shows how different income deciles benefit from the reform, with both absolute (£) and relative (%) changes

The reform creates a permanent 2% increase in rail subsidies that compounds through all future years, providing additional household income support proportional to rail usage.

In [11]:
# Calculate aggregates using household weights
total_households = (baseline_decile_df['household_weight']).sum()
total_rail_subsidy_gain = (baseline_decile_df['rail_subsidy_gain'] * baseline_decile_df['household_weight']).sum()
total_income_change = (baseline_decile_df['income_change'] * baseline_decile_df['household_weight']).sum()

print("=" * 70)
print("AGGREGATE CHANGES (using household weights)")
print("=" * 70)
print(f"Total households (deciles 1-10): {total_households:,.0f}")
print(f"Rail subsidy gain:      £{total_rail_subsidy_gain / 1e9:,.2f} bn")
print(f"Income change:          £{total_income_change / 1e9:,.2f} bn")
print(f"\nVerification: Income change = Rail subsidy gain? {'✓ Yes' if abs(total_income_change - total_rail_subsidy_gain) < 1 else '✗ No'}")

AGGREGATE CHANGES (using household weights)
Total households (deciles 1-10): 31,480,496
Rail subsidy gain:      £0.15 bn
Income change:          £0.15 bn

Verification: Income change = Rail subsidy gain? ✓ Yes


## Methodology Note: Defense of the 2% Reform Multiplier

### Model Specification

This analysis applies a **2% multiplier** to baseline government rail subsidy in 2026, representing additional public spending to compensate for the rail fare freeze policy.

### External Evidence Base

#### 1. Rail Funding Structure (ORR, 2024/25)
- **Government subsidy: £11.9bn** (~50% of rail industry income)
- **Passenger fares: £11.5bn** (~50% of rail industry income)
- **Source**: [Office of Rail and Road](https://www.orr.gov.uk/search-news/increasing-fares-revenue-contributes-reduction-government-subsidy-railways)

#### 2. Regulated Fares Coverage (GOV.UK)
- **Only 45% of rail fares are regulated** by government
- Includes season tickets, day singles/returns, off-peak and peak returns between major cities
- **Source**: [GOV.UK Rail Fares Freeze Announcement](https://www.gov.uk/government/publications/rail-fares-freeze-passenger-savings-estimate)

#### 3. Counterfactual Fare Increase (GOV.UK)
- Without freeze, regulated fares would increase by **5.8%** (July 2025 RPI of 4.8% + 1%)
- **Source**: [GOV.UK Rail Fares Freeze Document](https://www.gov.uk/government/publications/rail-fares-freeze-passenger-savings-estimate)

#### 4. Passenger Savings (GOV.UK)
- Fare freeze expected to save passengers **£600 million** in 2026/27
- **Source**: [GOV.UK Official Document](https://www.gov.uk/government/publications/rail-fares-freeze-passenger-savings-estimate)

#### 5. Net Fiscal Cost (GOV.UK)
Government states:
> "The net cost of the policy will be **significantly lower than** £600 million because the freeze is assumed to stimulate new journeys."

This acknowledges **behavioral offsets** from increased ridership generating additional fare revenue.

**Source**: [GOV.UK Rail Fares Freeze](https://www.gov.uk/government/publications/rail-fares-freeze-passenger-savings-estimate)

### Justification of 2% Multiplier

The **2% multiplier** is justified by:

1. **Partial coverage**: Only 45% of fares are regulated (GOV.UK)
2. **50:50 funding split**: Rail spending is approximately half subsidy, half fares (ORR)
3. **Behavioral offsets**: Government expects net cost to be "significantly lower" than gross savings (GOV.UK)
4. **Conservative estimate**: 2% represents a policy-calibrated estimate reflecting:
   - Limited scope of regulated fares
   - Revenue generation from increased ridership
   - Operational efficiency adjustments

### Distributional Approach

The additional government subsidy is distributed proportionally to households based on their baseline `rail_subsidy_spending`, which serves as a proxy for:
- Rail usage intensity
- Exposure to regulated fares
- Geographic commuting patterns

This ensures households who use rail services more (and thus benefit more from the freeze) receive proportionally larger income gains.